<a href="https://colab.research.google.com/github/ZataraHere/Basic-Neural-Networks/blob/main/DNN_with_softmax_layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import numpy as np
from tensorflow.keras.datasets import mnist

In [2]:
class DeepNeuralNetwork:

    def __init__(self, layer_dims, learning_rate=0.01):

        self.layer_dims = layer_dims
        self.learning_rate = learning_rate

        self.parameters = {}

        self.initialize_parameters()

    # ==================================================
    # Initialization
    # ==================================================

    def initialize_parameters(self):

        np.random.seed(42)

        L = len(self.layer_dims)

        for l in range(1, L):

            fan_in = self.layer_dims[l - 1]

            self.parameters[f"W{l}"] = (
                np.random.randn(
                    self.layer_dims[l],
                    fan_in
                )
                * np.sqrt(2.0 / fan_in)
            )

            self.parameters[f"b{l}"] = np.zeros(
                (self.layer_dims[l], 1)
            )

    # ==================================================
    # Activations
    # ==================================================

    def softmax(self, Z):

        # Numerical stability
        Z = Z - np.max(
            Z,
            axis=0,
            keepdims=True
        )

        exp_Z = np.exp(Z)

        return exp_Z / np.sum(
            exp_Z,
            axis=0,
            keepdims=True
        )

    def relu(self, Z):

        return np.maximum(0, Z)

    def relu_derivative(self, Z):

        return (Z > 0).astype(float)

    # ==================================================
    # Forward propagation
    # ==================================================

    def forward_propagation(self, X):

        cache = []

        A_prev = X

        L = len(self.layer_dims) - 1

        for l in range(1, L + 1):

            W = self.parameters[f"W{l}"]
            b = self.parameters[f"b{l}"]

            Z = W @ A_prev + b

            # Hidden layers
            if l < L:

                A = self.relu(Z)

            # Output layer
            else:

                A = self.softmax(Z)

            cache.append(
                (A_prev, W, b, Z, A)
            )

            A_prev = A

        return A_prev, cache

    # ==================================================
    # Categorical Cross Entropy
    # ==================================================

    def compute_cost(self, AL, Y):

        m = Y.shape[1]

        AL = np.clip(
            AL,
            1e-15,
            1.0
        )

        cost = -(
            1 / m
        ) * np.sum(
            Y * np.log(AL)
        )

        return np.squeeze(cost)

    # ==================================================
    # Backpropagation
    # ==================================================

    def backward_propagation(self, AL, Y, cache):

        grads = {}

        m = Y.shape[1]

        L = len(cache)

        # Make sure Y has same shape as AL
        Y = Y.reshape(AL.shape)

        # ==================================================
        # Output layer
        # ==================================================

        A_prev, W, b, Z, A = cache[L - 1]

        # Softmax + Cross Entropy
        #
        # dZ = AL - Y
        #

        dZ = AL - Y

        grads[f"dW{L}"] = (
            dZ @ A_prev.T
        ) / m

        grads[f"db{L}"] = (
            np.sum(
                dZ,
                axis=1,
                keepdims=True
            )
        ) / m

        # Gradient flowing into previous layer
        dA_prev = W.T @ dZ

        # ==================================================
        # Hidden layers
        # ==================================================

        for l in reversed(range(L - 1)):

            A_prev, W, b, Z, A = cache[l]

            dZ = (
                dA_prev
                * self.relu_derivative(Z)
            )

            grads[f"dW{l + 1}"] = (
                dZ @ A_prev.T
            ) / m

            grads[f"db{l + 1}"] = (
                np.sum(
                    dZ,
                    axis=1,
                    keepdims=True
                )
            ) / m

            dA_prev = W.T @ dZ

        return grads

    # ==================================================
    # Parameter update
    # ==================================================

    def update_parameters(self, grads):

        L = len(self.layer_dims) - 1

        for l in range(1, L + 1):

            self.parameters[f"W{l}"] -= (
                self.learning_rate
                * grads[f"dW{l}"]
            )

            self.parameters[f"b{l}"] -= (
                self.learning_rate
                * grads[f"db{l}"]
            )

    # ==================================================
    # Training
    # ==================================================

    def fit(
        self,
        X,
        Y,
        epochs=1000,
        print_cost=True
    ):

        for i in range(epochs):

            # Forward
            AL, cache = self.forward_propagation(X)

            # Cost
            cost = self.compute_cost(
                AL,
                Y
            )

            # Backward
            grads = self.backward_propagation(
                AL,
                Y,
                cache
            )

            # Update
            self.update_parameters(grads)

            if (
                print_cost
                and i % max(epochs // 10, 1) == 0
            ):

                print(
                    f"Cost after epoch {i}: {cost:.4f}"
                )

    # ==================================================
    # Prediction
    # ==================================================

    def predict(self, X):

        AL, _ = self.forward_propagation(X)

        # Return class index
        return np.argmax(
            AL,
            axis=0
        )


In [3]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("Original X_train:", X_train.shape)
print("Original y_train:", y_train.shape)


# ============================================================
# Flatten images
# ============================================================

# (60000, 28, 28)
#       ↓
# (60000, 784)

X_train = X_train.reshape(
    X_train.shape[0],
    -1
)

X_test = X_test.reshape(
    X_test.shape[0],
    -1
)


# ============================================================
# Normalize pixels
# ============================================================

X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0


# ============================================================
# Transpose
# ============================================================

# Your network expects:
#
# (features, examples)
#
# Therefore:
#
# X_train -> (784, 60000)

X_train = X_train.T
X_test = X_test.T


# ============================================================
# One-hot encode labels
# ============================================================

num_classes = 10

Y_train = np.eye(num_classes)[y_train]
Y_test = np.eye(num_classes)[y_test]


# Transpose:
#
# (60000, 10)
#      ↓
# (10, 60000)

Y_train = Y_train.T
Y_test = Y_test.T


print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_test:", X_test.shape)
print("Y_test:", Y_test.shape)


Original X_train: (60000, 28, 28)
Original y_train: (60000,)
X_train: (784, 60000)
Y_train: (10, 60000)
X_test: (784, 10000)
Y_test: (10, 10000)


In [4]:
model = DeepNeuralNetwork(
    layer_dims=[
        784,    # input
        128,    # hidden
        64,     # hidden
        32,     # hidden
        10      # output
    ],
    learning_rate=0.01
)


In [9]:
model.fit(
    X_train,
    Y_train,
    epochs=500,
    print_cost=True
)


Cost after epoch 0: 1.7214
Cost after epoch 50: 1.3373
Cost after epoch 100: 1.0316
Cost after epoch 150: 0.8314
Cost after epoch 200: 0.7050
Cost after epoch 250: 0.6204
Cost after epoch 300: 0.5601
Cost after epoch 350: 0.5152
Cost after epoch 400: 0.4804
Cost after epoch 450: 0.4527


In [10]:
predictions = model.predict(X_test)

print(predictions[:20])
print(y_test[:20])


[7 2 1 0 4 1 4 9 2 9 0 2 9 0 1 5 9 7 3 4]
[7 2 1 0 4 1 4 9 5 9 0 6 9 0 1 5 9 7 3 4]


In [11]:
def accuracy(predictions, labels):
    return np.mean(predictions == labels)


accuracy_score = accuracy(
    predictions,
    y_test
)

print(f"Accuracy: {accuracy_score * 100:.2f}%")


Accuracy: 88.89%
